In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import plotly.io as pio
from idea.profile.profile import calculate_profile
from idea.validation.validation import validate_roadwork

from util.visualization import create_multiplot_heatmap, create_plot_by_segment

pd.options.plotting.backend = "plotly"
pio.renderers.default = "colab"

## Example Validations

This repository contains three example cases:

- **Example 1**:
  There is always the maximum amount of FCD coverage during the profile period.
  Even during the roadworks, coverage does not decrease.
  **→ Status: `open`**

- **Example 2**:
  There is always the maximum amount of FCD coverage during the profile period.
  During the roadworks, there is no coverage, but occasionally a vehicle is detected.
  **→ Status: `closed` during periods with less than usual coverage**

- **Example 3**:
  There is consistently very low FCD coverage during the profile period.
  During the roadworks, coverage drops to none, with occasional vehicle detection.
  There is one short period where coverage increases, and the algorithm marks it as `open`.
  **→ Status: `closed`, with one short `open` period due to increased coverage**



In [ ]:
# Choose the test you want to run (1 till 3 available)
EXAMPLE_FOLDER = "example2"

In [ ]:
# Retrieve fcd during profile period.
df_minute = pd.read_csv(rf"resources/{EXAMPLE_FOLDER}/fcd_profile_input.csv", sep=";", index_col=0)
df_minute.index = pd.to_datetime(df_minute.index, utc=True)
start = pd.Timestamp("2024-01-01 00:00:00", tz="UTC")
end = pd.Timestamp("2025-01-01 00:00:00", tz="UTC")
profile = calculate_profile(df_minute, start, end)

# Retrieve fcd during roadwork.
validation_start = pd.Timestamp("2025-03-15 12:00:00", tz="UTC")
validation_end = pd.Timestamp("2025-03-15 15:00:00", tz="UTC")
fcd_during_roadwork = pd.read_csv(
    rf"resources/{EXAMPLE_FOLDER}/fcd_during_roadwork.csv", sep=";", index_col=0
)
fcd_during_roadwork.index = pd.to_datetime(fcd_during_roadwork.index, utc=True)

In [ ]:
validated_roadwork = validate_roadwork(fcd_during_roadwork, profile)

In [ ]:
validated_roadwork

In [ ]:
# Add order column (needed for plot function)
validated_roadwork["order"] = 1

In [ ]:
validated_roadwork.head()

## Visualization of the validation values for a single segment.

In [ ]:
fig = create_multiplot_heatmap(
    validated_roadwork,
    [
        "fcd",
        "running_mean",
        "segment_closure_status",
        "fcd_mean_median",
        "max_consecutive_zeros_q95",
        "max_consecutive_zeros_or_ones_q95",
    ],
)
fig.show()

## Visualization of the running mean for a single segment

In [ ]:
fig = create_plot_by_segment(validated_roadwork)
fig.show()